# DevSecOps Remediation Agent — API Integration Test Harness

**Purpose:** exercise every external integration (GitHub, Prisma report parsing, Artifactory,
Anthropic, Teams) against **dummy fixture files** instead of live services, so you can validate
the pipeline logic before pointing it at a real repo.

**How it works:**
- All the pipeline logic (parsing, decision engine, comparison, etc.) is the *real* code from the
  production notebook — nothing about the logic itself is faked.
- Every external API client (`GitHubClient`, `ArtifactoryClient`, `AnthropicClient`, `TeamsClient`)
  is swapped for a **mock implementation** that returns canned responses from fixture files and
  **records every call** it received, so you can assert on what the pipeline *would have* sent to
  the real API — without needing any credentials.
- Fixtures are plain JSON/YAML/Dockerfile files. Point the loader at your own files, or use the
  bundled dummy set (several Prisma reports at different severities, two Dockerfiles, one image
  manifest).

**To use your own dummy files:** upload them, then change the `FIXTURE_DIR` / individual path
variables in Phase 2 to point at `/mnt/user-data/uploads/...`.


---
## Phase 1 — Setup


In [ ]:
import os, re, json, yaml, dataclasses, logging
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Optional, TypedDict, Any, Callable

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("test-harness")

FIXTURE_DIR = "fixtures"   # change to /mnt/user-data/uploads to use your own files
os.makedirs(FIXTURE_DIR, exist_ok=True)


---
## Phase 2 — Dummy Fixture Files

Five Prisma report fixtures (clean, medium-only, high, critical, malformed-for-error-testing),
two Dockerfiles (good/bad), and one image manifest. Written to `fixtures/` so this notebook is
runnable standalone — replace any of these with your own uploaded files at any time.


In [ ]:
FIXTURES = {
    "prisma_report_critical.json": {
        "results": [{
            "name": "myregistry.io/payments-api:1.4.2",
            "vulnerabilities": [
                {"id": "CVE-2024-6119", "severity": "critical", "packageName": "openssl",
                 "packageVersion": "3.0.2", "fixDate": "3.0.13", "cvss": 9.8,
                 "description": "OpenSSL denial of service via crafted X.509 certificate."},
                {"id": "CVE-2023-44487", "severity": "high", "packageName": "nghttp2",
                 "packageVersion": "1.43.0", "fixDate": "1.57.0", "cvss": 7.5,
                 "description": "HTTP/2 Rapid Reset DoS."},
                {"id": "CVE-2022-37434", "severity": "medium", "packageName": "zlib",
                 "packageVersion": "1.2.11", "fixDate": "1.2.12", "cvss": 5.5,
                 "description": "Heap buffer over-read in inflate()."},
            ],
        }],
    },
    "prisma_report_high.json": {
        "results": [{
            "name": "myregistry.io/notifications-worker:2.0.0",
            "vulnerabilities": [
                {"id": "CVE-2023-44487", "severity": "high", "packageName": "nghttp2",
                 "packageVersion": "1.43.0", "fixDate": "1.57.0", "cvss": 7.5,
                 "description": "HTTP/2 Rapid Reset DoS."},
            ],
        }],
    },
    "prisma_report_medium.json": {
        "results": [{
            "name": "myregistry.io/notifications-worker:2.0.1",
            "vulnerabilities": [
                {"id": "CVE-2022-37434", "severity": "medium", "packageName": "zlib",
                 "packageVersion": "1.2.11", "fixDate": "1.2.12", "cvss": 5.5,
                 "description": "Heap buffer over-read in inflate()."},
            ],
        }],
    },
    "prisma_report_clean.json": {
        "results": [{"name": "myregistry.io/payments-api:1.4.3", "vulnerabilities": []}],
    },
    "prisma_report_malformed.json": {
        "results": [{"name": "myregistry.io/broken-image:0.0.1"}],  # missing "vulnerabilities" key entirely
    },
}

DOCKERFILES = {
    "Dockerfile.bad": """FROM myregistry.io/base-images/python:latest
RUN apt-get update && apt-get install -y curl openssl
COPY . /app
WORKDIR /app
RUN pip install -r requirements.txt
CMD ["python", "app.py"]
""",
    "Dockerfile.good": """FROM myregistry.io/base-images/python:3.11.9-slim
RUN apt-get update && apt-get install -y --no-install-recommends curl=8.5.0-2 \\
    && rm -rf /var/lib/apt/lists/*
COPY . /app
WORKDIR /app
RUN pip install --no-cache-dir -r requirements.txt
USER appuser
CMD ["python", "app.py"]
""",
}

IMAGE_MANIFEST_YAML = """images:
  - name: payments-api
    current_image: myregistry.io/payments-api:1.4.2
    dockerfile_path: services/payments-api/Dockerfile
    owner_team: payments
    environment: production
  - name: notifications-worker
    current_image: myregistry.io/notifications-worker:2.0.0
    dockerfile_path: services/notifications-worker/Dockerfile
    owner_team: platform
    environment: production
"""

for fname, content in FIXTURES.items():
    with open(os.path.join(FIXTURE_DIR, fname), "w") as f:
        json.dump(content, f, indent=2)
for fname, content in DOCKERFILES.items():
    with open(os.path.join(FIXTURE_DIR, fname), "w") as f:
        f.write(content)
with open(os.path.join(FIXTURE_DIR, "images.yaml"), "w") as f:
    f.write(IMAGE_MANIFEST_YAML)

print("Fixtures written to:", os.path.abspath(FIXTURE_DIR))
print(sorted(os.listdir(FIXTURE_DIR)))


In [ ]:
def load_json_fixture(filename: str) -> dict:
    """Load a dummy Prisma report — swap in your own file path here (e.g. from /mnt/user-data/uploads)."""
    path = os.path.join(FIXTURE_DIR, filename) if not os.path.isabs(filename) else filename
    with open(path) as f:
        return json.load(f)

def load_text_fixture(filename: str) -> str:
    path = os.path.join(FIXTURE_DIR, filename) if not os.path.isabs(filename) else filename
    with open(path) as f:
        return f.read()

def load_image_manifest(filename: str = "images.yaml") -> list[dict]:
    return yaml.safe_load(load_text_fixture(filename))["images"]


---
## Phase 3 — Core Pipeline Logic (unchanged from production notebook)

This is the real logic — parsing, decision engine, comparison — copied verbatim from the
production notebook so this harness tests the actual code paths, not a re-implementation.


In [ ]:
SEVERITY_ORDER = {"critical": 4, "high": 3, "medium": 2, "low": 1, "unimportant": 0}

@dataclass
class Vulnerability:
    cve_id: str
    severity: str
    package_name: str
    package_version: str
    fix_version: Optional[str]
    description: str
    cvss: Optional[float] = None

    @property
    def rank(self) -> int:
        return SEVERITY_ORDER.get(self.severity.lower(), 0)


def parse_prisma_report(report: dict, image_name: str) -> list[Vulnerability]:
    vulns: list[Vulnerability] = []
    for result in report.get("results", []):
        if result.get("name") and image_name not in result.get("name", ""):
            continue
        for v in result.get("vulnerabilities", []) or []:   # tolerant of missing key -> []
            vulns.append(Vulnerability(
                cve_id=v.get("id", "UNKNOWN"),
                severity=v.get("severity", "unimportant"),
                package_name=v.get("packageName", "unknown"),
                package_version=v.get("packageVersion", "unknown"),
                fix_version=v.get("fixDate") or v.get("status"),
                description=v.get("description", ""),
                cvss=v.get("cvss"),
            ))
    vulns.sort(key=lambda v: v.rank, reverse=True)
    return vulns


def analyze_dockerfile(dockerfile_text: str) -> dict:
    base_image_match = re.search(r"^FROM\s+(\S+)", dockerfile_text, re.MULTILINE)
    base_image = base_image_match.group(1) if base_image_match else None
    suggestions = []
    if base_image and ":latest" in base_image:
        suggestions.append("Base image uses `:latest` — pin to an explicit, scanned tag/digest.")
    if not re.search(r"USER\s+(?!root)\S+", dockerfile_text):
        suggestions.append("No non-root USER directive found — container likely runs as root.")
    return {"base_image": base_image, "suggestions": suggestions}


class RemediationAction(str, Enum):
    SWAP_TO_ARTIFACTORY_IMAGE = "swap_to_artifactory_image"
    BUMP_BASE_IMAGE = "bump_base_image"
    PIN_PACKAGES = "pin_packages"
    MANUAL_REVIEW = "manual_review"


@dataclass
class Decision:
    action: RemediationAction
    target_image: Optional[str]
    reasoning: str
    blocking: bool


def heuristic_analysis(vulns: list[Vulnerability], threshold: str = "high") -> dict:
    by_pkg: dict[str, list[Vulnerability]] = {}
    for v in vulns:
        by_pkg.setdefault(v.package_name, []).append(v)
    clusters = []
    for pkg, vs in by_pkg.items():
        action = "bump_base_image" if pkg in {"openssl", "glibc", "libssl"} else "pin_package"
        clusters.append({"root_package": pkg, "cve_ids": [v.cve_id for v in vs],
                          "recommended_action": action})
    max_rank = max((v.rank for v in vulns), default=0)
    overall = {4: "critical", 3: "high", 2: "medium", 1: "low"}.get(max_rank, "low")
    return {"overall_risk": overall, "blocking": max_rank >= SEVERITY_ORDER[threshold], "clusters": clusters}


def decide(analysis: dict, secure_image_hit: Optional[dict], dockerfile_analysis: Optional[dict]) -> Decision:
    if secure_image_hit is not None:
        return Decision(RemediationAction.SWAP_TO_ARTIFACTORY_IMAGE,
                         secure_image_hit.get("path"), "Clean image already in Artifactory.",
                         analysis.get("blocking", False))
    bump = [c for c in analysis.get("clusters", []) if c["recommended_action"] == "bump_base_image"]
    if bump:
        return Decision(RemediationAction.BUMP_BASE_IMAGE, None,
                         f"Base-image-level fix needed for {bump[0]['root_package']}.",
                         analysis.get("blocking", False))
    pin = [c for c in analysis.get("clusters", []) if c["recommended_action"] == "pin_package"]
    if pin:
        return Decision(RemediationAction.PIN_PACKAGES, None,
                         f"{len(pin)} package(s) resolvable via version pin.",
                         analysis.get("blocking", False))
    return Decision(RemediationAction.MANUAL_REVIEW, None, "No confident automatic path.",
                     analysis.get("blocking", False))


def compare_scans(before: list[Vulnerability], after: list[Vulnerability]) -> dict:
    before_ids, after_ids = {v.cve_id for v in before}, {v.cve_id for v in after}
    return {
        "before_total": len(before), "after_total": len(after),
        "resolved_cve_ids": sorted(before_ids - after_ids),
        "new_cve_ids": sorted(after_ids - before_ids),
        "improved": len(after) < len(before),
    }


---
## Phase 4 — Mock API Clients

Same method signatures the real clients use, but backed by fixtures + an in-memory call log
instead of `requests.post`/`requests.get`. Assert against `.calls` in your tests.


In [ ]:
@dataclass
class RecordedCall:
    method: str
    args: tuple
    kwargs: dict
    response: Any


class MockGitHubClient:
    """Mimics github_client.py — no real network calls, just records + returns canned data."""
    def __init__(self):
        self.calls: list[RecordedCall] = []
        self._next_run_id = 1001
        self._prs_opened: list[dict] = []
        self._branches_created: list[str] = []

    def _record(self, method, args, kwargs, response):
        self.calls.append(RecordedCall(method, args, kwargs, response))
        return response

    def trigger_workflow(self, image: dict, ref: str = "main") -> dict:
        return self._record("trigger_workflow", (image, ref), {}, {"dispatched": True})

    def get_latest_run(self, image: dict) -> dict:
        run = {"id": self._next_run_id, "status": "completed", "conclusion": "success"}
        self._next_run_id += 1
        return self._record("get_latest_run", (image,), {}, run)

    def monitor_run(self, run_id: int) -> dict:
        return self._record("monitor_run", (run_id,), {}, {"id": run_id, "conclusion": "success"})

    def create_branch(self, image_name: str) -> str:
        branch = f"auto-remediate/{image_name}-TEST"
        self._branches_created.append(branch)
        return self._record("create_branch", (image_name,), {}, branch)

    def commit_file(self, branch: str, path: str, content: str, message: str) -> dict:
        return self._record("commit_file", (branch, path, message), {}, {"committed": True, "sha": "deadbeef"})

    def create_pull_request(self, branch: str, image: dict, decision: Decision) -> dict:
        pr = {"number": len(self._prs_opened) + 1,
              "html_url": f"https://github.com/org/repo/pull/{len(self._prs_opened) + 1}",
              "branch": branch}
        self._prs_opened.append(pr)
        return self._record("create_pull_request", (branch, image["name"], decision.action.value), {}, pr)


class MockArtifactoryClient:
    """Feed it a dict of {image_name_prefix: hit_or_None} to control hit/miss per test case."""
    def __init__(self, canned_results: dict[str, Optional[dict]] | None = None):
        self.calls: list[RecordedCall] = []
        self.canned_results = canned_results or {}

    def search_secure_image(self, image: dict) -> Optional[dict]:
        result = self.canned_results.get(image["name"])
        self.calls.append(RecordedCall("search_secure_image", (image["name"],), {}, result))
        return result


class MockAnthropicClient:
    """Wraps the real heuristic so tests don't need ANTHROPIC_API_KEY, but still 'looks like' an API call.
    NOTE: this does NOT exercise the actual LLM call, prompt, or response parsing -- see Phase 4b
    below for that. This mock only tests that the pipeline calls *something* at the right point
    and consumes its output correctly."""
    def __init__(self, threshold: str = "high"):
        self.calls: list[RecordedCall] = []
        self.threshold = threshold

    def analyze(self, vulns: list[Vulnerability]) -> dict:
        result = heuristic_analysis(vulns, self.threshold)
        self.calls.append(RecordedCall("analyze", (len(vulns),), {}, result))
        return result


class MockTeamsClient:
    def __init__(self):
        self.calls: list[RecordedCall] = []
        self.sent_cards: list[dict] = []

    def send(self, card: dict) -> dict:
        self.sent_cards.append(card)
        return self._record_and_return(card)

    def _record_and_return(self, card):
        self.calls.append(RecordedCall("send", (card.get("title"),), {}, {"status_code": 200}))
        return {"status_code": 200}


---
## Phase 4b — Real LLM / Agent Integration Test

This is the part that actually exercises Claude. Everything above tests the *pipeline*; this
tests the *AI agent itself*: the system prompt, the real API round-trip, and — just as
importantly — whether the response parsing survives the LLM **not** returning perfectly clean
JSON (markdown fences, trailing prose, truncated output), which is the failure mode that breaks
this kind of integration in production.

Runs live against Claude if `ANTHROPIC_API_KEY` is set; otherwise runs the parsing/validation
logic against canned dummy LLM responses so you can still test that path without a key.


In [ ]:
ANALYSIS_SYSTEM_PROMPT = """You are a container security triage engine. You will be given a JSON list of \
CVEs found in a container image. Respond with ONLY valid JSON (no markdown fences, no prose) matching \
this schema:
{
  "overall_risk": "critical|high|medium|low",
  "blocking": true|false,
  "clusters": [
    {"root_package": str, "cve_ids": [str], "recommended_action": "bump_base_image|pin_package|rewrite_dockerfile",
     "rationale": str}
  ],
  "summary": str
}
"""

REQUIRED_KEYS = {"overall_risk", "blocking", "clusters", "summary"}
VALID_RISK = {"critical", "high", "medium", "low"}
VALID_ACTIONS = {"bump_base_image", "pin_package", "rewrite_dockerfile"}


class LLMResponseError(ValueError):
    """Raised when the model's output doesn't match the contract the decision engine relies on."""


def extract_json_from_llm_text(text: str) -> dict:
    """LLMs sometimes wrap JSON in ```json fences or add a stray sentence -- strip defensively
    before parsing instead of assuming clean output. This is the #1 real-world failure point."""
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned)
        cleaned = re.sub(r"```\s*$", "", cleaned)
    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if not match:
        raise LLMResponseError(f"No JSON object found in LLM response: {text[:200]!r}")
    try:
        return json.loads(match.group(0))
    except json.JSONDecodeError as e:
        raise LLMResponseError(f"LLM response was not valid JSON: {e}") from e


def validate_llm_analysis_schema(data: dict) -> list[str]:
    """Returns a list of schema problems (empty list = valid). Never raises -- the decision engine
    should be able to check this and fall back to the heuristic instead of crashing on bad output."""
    problems = []
    missing = REQUIRED_KEYS - data.keys()
    if missing:
        problems.append(f"missing keys: {missing}")
    if "overall_risk" in data and data["overall_risk"] not in VALID_RISK:
        problems.append(f"invalid overall_risk: {data['overall_risk']!r}")
    if "blocking" in data and not isinstance(data["blocking"], bool):
        problems.append(f"blocking should be bool, got {type(data['blocking']).__name__}")
    for i, cluster in enumerate(data.get("clusters", [])):
        if cluster.get("recommended_action") not in VALID_ACTIONS:
            problems.append(f"clusters[{i}].recommended_action invalid: {cluster.get('recommended_action')!r}")
    return problems


class RealAnthropicClient:
    """Actual Anthropic API client -- calls Claude for real when ANTHROPIC_API_KEY is set."""
    def __init__(self, model: str = "claude-sonnet-4-6"):
        self.model = model
        self.calls: list[RecordedCall] = []
        self._client = None
        if os.environ.get("ANTHROPIC_API_KEY"):
            from anthropic import Anthropic
            self._client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

    def analyze(self, vulns: list[Vulnerability], image_name: str = "test-image") -> dict:
        if self._client is None:
            raise RuntimeError("No ANTHROPIC_API_KEY set -- use analyze_from_canned_response() "
                                "below to test the parsing/validation path without a live call.")
        payload = [dataclasses.asdict(v) for v in vulns]
        msg = self._client.messages.create(
            model=self.model, max_tokens=1500, system=ANALYSIS_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": json.dumps({"image": image_name, "cves": payload})}],
        )
        raw_text = "".join(b.text for b in msg.content if b.type == "text")
        self.calls.append(RecordedCall("analyze", (image_name, len(vulns)), {}, raw_text))
        data = extract_json_from_llm_text(raw_text)
        problems = validate_llm_analysis_schema(data)
        if problems:
            raise LLMResponseError(f"LLM response failed schema validation: {problems}")
        return data


In [ ]:
# --- Canned dummy LLM responses (good, fenced, and malformed) to test the parser without a live call ---
CANNED_LLM_RESPONSES = {
    "clean_json": '''{"overall_risk": "critical", "blocking": true, "clusters": [
        {"root_package": "openssl", "cve_ids": ["CVE-2024-6119"], "recommended_action": "bump_base_image",
         "rationale": "Critical DoS in cert parsing."}], "summary": "1 critical CVE in openssl."}''',

    "markdown_fenced": '''```json
{"overall_risk": "high", "blocking": true, "clusters": [
    {"root_package": "nghttp2", "cve_ids": ["CVE-2023-44487"], "recommended_action": "pin_package",
     "rationale": "Rapid reset DoS, patched upstream."}], "summary": "1 high CVE."}
```''',

    "chatty_prose": '''Sure, here is my analysis of the vulnerabilities you provided:

{"overall_risk": "medium", "blocking": false, "clusters": [
    {"root_package": "zlib", "cve_ids": ["CVE-2022-37434"], "recommended_action": "pin_package",
     "rationale": "Low exploitability heap over-read."}], "summary": "1 medium CVE."}

Let me know if you need anything else!''',

    "truncated_invalid": '''{"overall_risk": "high", "blocking": true, "clusters": [
        {"root_package": "openssl", "cve_ids": ["CVE-2024-6119"''',   # cut off mid-response

    "wrong_enum_value": '''{"overall_risk": "super-bad", "blocking": "yes", "clusters": [
        {"root_package": "openssl", "cve_ids": ["CVE-2024-6119"], "recommended_action": "just fix it",
         "rationale": "..."}], "summary": "..."}''',
}


def analyze_from_canned_response(response_key: str) -> dict:
    """Runs the exact same parse+validate path the real client uses, against a canned string --
    this is how you test the LLM integration contract without spending API calls or needing a key."""
    raw_text = CANNED_LLM_RESPONSES[response_key]
    data = extract_json_from_llm_text(raw_text)
    problems = validate_llm_analysis_schema(data)
    if problems:
        raise LLMResponseError(f"LLM response failed schema validation: {problems}")
    return data


for key in ["clean_json", "markdown_fenced", "chatty_prose"]:
    result = analyze_from_canned_response(key)
    print(f"[{key}] parsed OK -> overall_risk={result['overall_risk']}, blocking={result['blocking']}")

for key in ["truncated_invalid", "wrong_enum_value"]:
    try:
        analyze_from_canned_response(key)
        print(f"[{key}] UNEXPECTEDLY PARSED -- should have raised")
    except LLMResponseError as e:
        print(f"[{key}] correctly rejected -> {e}")


In [ ]:
def test_clean_json_parses_and_validates():
    r = analyze_from_canned_response("clean_json")
    assert r["overall_risk"] == "critical"

def test_markdown_fenced_response_is_stripped_correctly():
    r = analyze_from_canned_response("markdown_fenced")
    assert r["overall_risk"] == "high"

def test_chatty_prose_around_json_is_ignored():
    r = analyze_from_canned_response("chatty_prose")
    assert r["overall_risk"] == "medium"

def test_truncated_json_raises_llm_response_error():
    try:
        analyze_from_canned_response("truncated_invalid")
        assert False, "should have raised LLMResponseError"
    except LLMResponseError:
        pass

def test_invalid_enum_values_are_rejected_by_schema_check():
    try:
        analyze_from_canned_response("wrong_enum_value")
        assert False, "should have raised LLMResponseError"
    except LLMResponseError:
        pass

def test_real_client_requires_api_key_or_raises_clear_error():
    client = RealAnthropicClient()
    if client._client is None:
        try:
            client.analyze([], "test-image")
            assert False, "should have raised RuntimeError without a key"
        except RuntimeError as e:
            assert "ANTHROPIC_API_KEY" in str(e)
    else:
        print("  (ANTHROPIC_API_KEY is set -- skipping the no-key error-path assertion)")

def _run_llm_integration_tests():
    names = ["test_clean_json_parses_and_validates", "test_markdown_fenced_response_is_stripped_correctly",
             "test_chatty_prose_around_json_is_ignored", "test_truncated_json_raises_llm_response_error",
             "test_invalid_enum_values_are_rejected_by_schema_check",
             "test_real_client_requires_api_key_or_raises_clear_error"]
    for name in names:
        globals()[name]()
        print(f"PASS: {name}")
    print(f"\n{len(names)}/{len(names)} LLM-integration tests passed")

_run_llm_integration_tests()


In [ ]:
# --- Optional: run this cell for a real, live call to Claude (only fires if ANTHROPIC_API_KEY is set) ---
live_client = RealAnthropicClient()
if live_client._client is not None:
    critical_vulns = parse_prisma_report(load_json_fixture("prisma_report_critical.json"), "payments-api")
    live_result = live_client.analyze(critical_vulns, image_name="payments-api")
    print(json.dumps(live_result, indent=2))
else:
    print("No ANTHROPIC_API_KEY set -- skipping live call. "
          "The parsing/validation logic above was already tested against canned responses.")


---
## Phase 5 — Test Runner: Pipeline Against a Dummy Report

Runs the real parse → analyze → branch → decide → notify flow, entirely against fixture files
and mock clients. Change `report_file` / `dockerfile_file` to point at any dummy file you like.


In [ ]:
def run_pipeline_on_fixture(
    image: dict,
    report_file: str,
    dockerfile_file: Optional[str] = None,
    artifactory_hit: Optional[dict] = None,
    severity_threshold: str = "high",
) -> dict:
    """Runs one pipeline pass against dummy files. Returns everything a test would want to assert on."""
    gh = MockGitHubClient()
    art = MockArtifactoryClient(canned_results={image["name"]: artifactory_hit})
    ai = MockAnthropicClient(threshold=severity_threshold)
    teams = MockTeamsClient()

    # --- Phases 3-6: trigger, monitor, download(=load fixture), parse ---
    gh.trigger_workflow(image)
    run = gh.get_latest_run(image)
    gh.monitor_run(run["id"])
    report = load_json_fixture(report_file)
    vulns = parse_prisma_report(report, image["name"])

    # --- Phase 7: AI analysis ---
    analysis = ai.analyze(vulns)

    # --- Phase 8: branch ---
    hit = art.search_secure_image(image)
    dockerfile_analysis = None
    if hit is None and dockerfile_file:
        dockerfile_analysis = analyze_dockerfile(load_text_fixture(dockerfile_file))

    # --- Phase 9-13: decide, update config, branch/commit/PR ---
    decision = decide(analysis, hit, dockerfile_analysis)
    branch = gh.create_branch(image["name"])
    gh.commit_file(branch, "config/images.yaml", "...", f"chore: {decision.action.value}")
    pr = gh.create_pull_request(branch, image, decision)

    # --- Phase 16: notify ---
    teams.send({"title": f"Remediation — {image['name']}", "summary": decision.reasoning})

    return {
        "vulns": vulns, "analysis": analysis, "artifactory_hit": hit,
        "dockerfile_analysis": dockerfile_analysis, "decision": decision,
        "pr": pr, "clients": {"github": gh, "artifactory": art, "anthropic": ai, "teams": teams},
    }


demo_image = {"name": "payments-api", "current_image": "myregistry.io/payments-api:1.4.2"}

result = run_pipeline_on_fixture(demo_image, "prisma_report_critical.json", "Dockerfile.bad")
print("Decision:", result["decision"])
print("PR opened:", result["pr"])
print("GitHub calls made:", [c.method for c in result["clients"]["github"].calls])


---
## Phase 6 — Scenario Matrix: Run Every Dummy Report Through the Pipeline


In [ ]:
# NOTE: parse_prisma_report filters vulnerabilities by matching the image name against the
# report's "results[].name" field (mirrors real Prisma behavior when a report covers multiple
# images). Each scenario below uses the image whose name actually appears in that report fixture
# -- mixing an unrelated image name in here would silently zero out the vuln count.
notifications_image = {"name": "notifications-worker", "current_image": "myregistry.io/notifications-worker:2.0.0"}

SCENARIOS = [
    {"label": "Critical CVEs, no Artifactory hit -> Dockerfile path", "image": demo_image,
     "report_file": "prisma_report_critical.json", "dockerfile_file": "Dockerfile.bad", "artifactory_hit": None},
    {"label": "High CVEs, Artifactory has a clean image -> swap", "image": notifications_image,
     "report_file": "prisma_report_high.json", "dockerfile_file": "Dockerfile.bad",
     "artifactory_hit": {"path": "secure/notifications-worker:2.0.0-hardened"}},
    {"label": "Medium only, no hit -> package pin", "image": notifications_image,
     "report_file": "prisma_report_medium.json", "dockerfile_file": "Dockerfile.good", "artifactory_hit": None},
    {"label": "Clean report -> manual review (nothing to fix)", "image": demo_image,
     "report_file": "prisma_report_clean.json", "dockerfile_file": "Dockerfile.good", "artifactory_hit": None},
]

for s in SCENARIOS:
    r = run_pipeline_on_fixture(s["image"], s["report_file"], s["dockerfile_file"], s["artifactory_hit"])
    print(f"--- {s['label']} ---")
    print(f"  CVEs found: {len(r['vulns'])} | overall_risk: {r['analysis']['overall_risk']} "
          f"| action: {r['decision'].action.value}")
    print()


---
## Phase 7 — Error-Path Testing: Malformed / Missing Data

Real fixture data breaks in real ways. This section deliberately feeds bad input and checks the
pipeline degrades safely instead of throwing.


In [ ]:
def test_malformed_report_does_not_crash():
    report = load_json_fixture("prisma_report_malformed.json")  # no "vulnerabilities" key
    vulns = parse_prisma_report(report, "broken-image")
    assert vulns == [], "Missing vulnerabilities key should parse to an empty list, not raise"


def test_empty_dockerfile_returns_suggestions_not_error():
    result = analyze_dockerfile("")  # totally empty file
    assert result["base_image"] is None
    assert isinstance(result["suggestions"], list)


def test_unknown_severity_string_ranks_lowest():
    v = Vulnerability("CVE-TEST", "banana", "pkg", "1.0", None, "desc")
    assert v.rank == 0  # unrecognized severity should not crash SEVERITY_ORDER lookup


def test_pipeline_survives_clean_report_with_no_clusters():
    r = run_pipeline_on_fixture(demo_image, "prisma_report_clean.json", "Dockerfile.good")
    assert r["decision"].action == RemediationAction.MANUAL_REVIEW
    assert len(r["vulns"]) == 0


---
## Phase 8 — Assertions on Mock Call Logs

This is the part that actually validates "API integration" — checking the pipeline called the
right client methods, in the right order, with the right arguments, without ever hitting the
network.


In [ ]:
def test_swap_action_does_not_call_dockerfile_analysis():
    r = run_pipeline_on_fixture(demo_image, "prisma_report_high.json", "Dockerfile.bad",
                                 artifactory_hit={"path": "secure/foo:hardened"})
    assert r["dockerfile_analysis"] is None
    assert r["decision"].action == RemediationAction.SWAP_TO_ARTIFACTORY_IMAGE


def test_pr_is_only_opened_once_per_run():
    r = run_pipeline_on_fixture(demo_image, "prisma_report_critical.json", "Dockerfile.bad")
    pr_calls = [c for c in r["clients"]["github"].calls if c.method == "create_pull_request"]
    assert len(pr_calls) == 1


def test_teams_notification_sent_exactly_once():
    r = run_pipeline_on_fixture(demo_image, "prisma_report_critical.json", "Dockerfile.bad")
    assert len(r["clients"]["teams"].sent_cards) == 1


def test_workflow_triggered_before_pr_opened():
    r = run_pipeline_on_fixture(demo_image, "prisma_report_critical.json", "Dockerfile.bad")
    methods_in_order = [c.method for c in r["clients"]["github"].calls]
    assert methods_in_order.index("trigger_workflow") < methods_in_order.index("create_pull_request")


def _run_all_tests():
    tests = [v for k, v in globals().items() if k.startswith("test_")]
    for t in tests:
        t()
        print(f"PASS: {t.__name__}")
    print(f"\n{len(tests)}/{len(tests)} tests passed")

_run_all_tests()


---
## Phase 9 — Bring Your Own Dummy File

To test against a report or Dockerfile you provide instead of the bundled fixtures:

1. Upload the file in chat (or place it under `/mnt/user-data/uploads/`).
2. Run the cell below with the path — it works with any Prisma-shaped JSON report or Dockerfile.


In [ ]:
def run_pipeline_on_custom_file(image: dict, report_path: str, dockerfile_path: str | None = None,
                                  artifactory_hit: Optional[dict] = None) -> dict:
    """Same as run_pipeline_on_fixture but accepts absolute paths (e.g. your uploaded files)."""
    gh, art, ai, teams = MockGitHubClient(), MockArtifactoryClient({image["name"]: artifactory_hit}), \
                          MockAnthropicClient(), MockTeamsClient()
    with open(report_path) as f:
        report = json.load(f)
    vulns = parse_prisma_report(report, image["name"])
    analysis = ai.analyze(vulns)
    hit = art.search_secure_image(image)
    dockerfile_analysis = None
    if hit is None and dockerfile_path:
        with open(dockerfile_path) as f:
            dockerfile_analysis = analyze_dockerfile(f.read())
    decision = decide(analysis, hit, dockerfile_analysis)
    return {"vulns": vulns, "analysis": analysis, "decision": decision}


# Example (uncomment and point at your own uploaded file):
# result = run_pipeline_on_custom_file(
#     demo_image,
#     report_path="/mnt/user-data/uploads/my_prisma_report.json",
#     dockerfile_path="/mnt/user-data/uploads/Dockerfile",
# )
# result
print("Ready — call run_pipeline_on_custom_file(image, report_path, dockerfile_path) with your own files.")
